# Anthropic API

Anthropic API from/to translation helpers

In [ ]:
#| default_exp anthropic

## Imports

In [ ]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fastspec.errors import api_error_from_event

from fastllm.types import *
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream

In [ ]:
#| hide
import yaml
from fastspec.spec import *
from fastspec.oapi import *
from fastcore.test import *
from toolslm.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy
from importlib.resources import files

In [ ]:
enable_cachy(hdrs=('content-type',))

15:27:20 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'


15:27:20 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


In [ ]:
specs_path = files('fastllm') / 'specs'
ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
ant_spec

SpecParser(base_url='https://api.anthropic.com', ops=47)

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})

What is the latest anthropic version

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

## Normalize

Helpers to transform API responses into the internal representation types

### Tool Calls

Normalize tool calls shape(s)

In [ ]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
#| export
ant_tc_types = ("tool_use", "server_tool_use", "mcp_tool_use")

def norm_tool_call(b):
    if b.get("type") in ant_tc_types:
        extra = {k:v for k,v in b.items() if k not in ("type","id","name","input")}
        return ToolCall(id=b["id"], name=b["name"], arguments=b.get("input") or {}, server=b.get("type")!="tool_use", extra=extra)

def norm_tool_calls(resp, delta=False):
    "Extract Anthropic tool_use blocks as normalized tool calls."
    out = []
    for b in resp.get("content", []):
        if tc:= norm_tool_call(b): out.append(tc)
    return out

User defined:

In [ ]:
mn,mt = 'claude-sonnet-4-20250514',200

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[{"name": "simple_add", "description": "add numbers","input_schema": sch['function']['parameters']}])
norm_tool_calls(resp)

[ToolCall(id='toolu_01ConK83KqJAFezUYCbQXFmw', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, extra={'caller': {'type': 'direct'}}),
 ToolCall(id='toolu_01GBeFMh8XUgQEVqfRD2CC17', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, extra={'caller': {'type': 'direct'}})]

Web search:

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
norm_tool_calls(resp)

[ToolCall(id='srvtoolu_01A6kzMiDX7ZGzsW1nPp5ana', name='web_search', arguments={'query': 'Istanbul weather today'}, server=True, extra={})]

MCP:

In [ ]:
norm_tool_calls({"content": [{"type": "mcp_tool_use", "id": "mcptoolu_abc123", "name": "get_weather", "input": {"city": "Istanbul"}, "server_name": "weather-server"}]})

[ToolCall(id='mcptoolu_abc123', name='get_weather', arguments={'city': 'Istanbul'}, server=True, extra={'server_name': 'weather-server'})]

### Usage

Normalize usage shape(s)

In [ ]:
#| export
def norm_usage(resp):
    "Normalize Anthropic usage shape."
    if not (usg:=resp.get("usage")): return None
    cached = int(usg.get("cache_read_input_tokens", 0) or 0)
    cache_creation = int(usg.get("cache_creation_input_tokens", 0) or 0)
    pt = int(usg.get("input_tokens", 0) or 0) + cached + cache_creation
    ct = int(usg.get("output_tokens", 0) or 0)
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=pt + ct,
                 cached_tokens=cached, cache_creation_tokens=cache_creation, reasoning_tokens=0, raw=usg)

def finalize_usage(usg, parts):
    "Adjust usage using finalized Anthropic content parts."
    if not usg: return usg
    rc = '\n'.join(p.text or '' for p in parts if p.type == PartType.thinking)
    ct = int(usg.raw.get('output_tokens', usg.completion_tokens) or 0)
    rt = min(int(len(rc.split())*1.5), ct) if rc else 0
    return Usage(prompt_tokens=usg.prompt_tokens, completion_tokens=ct-rt, total_tokens=usg.prompt_tokens+ct,
        cached_tokens=usg.cached_tokens, cache_creation_tokens=usg.cache_creation_tokens, reasoning_tokens=rt, raw=usg.raw)

Text response:

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Hi"}])
norm_usage(resp)

Usage(prompt_tokens=8, completion_tokens=20, total_tokens=28, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 8, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 20, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Tool call response:

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}])
norm_usage(resp)

Usage(prompt_tokens=417, completion_tokens=138, total_tokens=555, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 417, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 138, 'service_tier': 'standard', 'inference_geo': 'not_available'})

Server tools:

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
norm_usage(resp)

Usage(prompt_tokens=11418, completion_tokens=255, total_tokens=11673, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 11418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 255, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})

### Finish Reason

Normalize finish reason shape(s)

In [ ]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = resp.get("stop_reason")
    mp = dict(end_turn=FinishReason.stop, tool_use=FinishReason.tool_calls, max_tokens=FinishReason.length)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [ ]:
norm_finish(resp)

<finish_reason.length: 'length'>

### Non-stream Completion

Normalize response into Completion.

In [ ]:
#| export
def _ant_part_type(typ):
    "Map Anthropic content block type to canonical PartType."
    ant_parts_mapping = dict(text=PartType.text, thinking=PartType.thinking, redacted_thinking=PartType.thinking, tool_use=PartType.tool_use, server_tool_use=PartType.tool_use, mcp_tool_use=PartType.tool_use)
    if typ in ant_parts_mapping: return ant_parts_mapping[typ]
    if typ.endswith('_tool_result'): return PartType.server_tool_result
    return typ

def norm_parts(resp):
    tcs = norm_tool_calls(resp)
    tc_map = {tc.id: tc for tc in tcs}
    parts = []
    for b in resp.get("content", []):
        typ = _ant_part_type(b.get("type", "text"))
        if typ == PartType.thinking: parts.append(Part(type=PartType.thinking, text=b.get("thinking", ""), data=b))
        elif typ == "tool_use":
            if tc:=tc_map.get(b.get("id")):
                tdata = {**tc.extra, 'id':tc.id, 'name':tc.name, 'arguments':tc.arguments, 'server':tc.server}
                parts.append(Part(type=PartType.tool_use, data=tdata))
        else: parts.append(Part(type=typ, text=b.get("text", ""), data=b))
    return parts

#### Text

In [ ]:
api_name,vnd_nm = 'anthropic', 'anthropic'

In [ ]:
# Not exported - later we add an extra param
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, finalize_usage=finalize_usage)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Say hi"}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

Hi there! How are you doing today?

<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=9, completion_tokens=12, total_tokens=21, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 9, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 12, 'service_tier': 'standard', 'inference_geo': 'not_available'})`

</details>

</div>

#### Thinking

In [ ]:
comp = mk_completion({"model": "claude-sonnet-4-20250514", "content": [
                  {"type": "thinking", "thinking": "Let me reason...", "text": ""},
                  {"type": "redacted_thinking", "text": ""},
                  {"type": "text", "text": "Here's my answer."}
], "stop_reason": "end_turn", "usage": {"input_tokens": 10, "output_tokens": 20}}, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

Let me reason...

</details>

Here's my answer.

<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=10, completion_tokens=16, total_tokens=30, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=4, raw={'input_tokens': 10, 'output_tokens': 20})`

</details>

</div>

#### Tool call

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}], tools=[{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

I'll calculate both additions using the simple_add tool in parallel.

🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=417, completion_tokens=138, total_tokens=555, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 417, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 138, 'service_tier': 'standard', 'inference_geo': 'not_available'})`

</details>

</div>

In [ ]:
test_eq(comp.tool_calls[0].server, False)

Web search response:

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, max_tokens=mt, messages=[{"role": "user", "content": "Use web search to get the weather in Istanbul"}], tools=[{"type": "web_search_20250305", "name": "web_search"}])
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

Based on the current weather search results for Istanbul, here's the weather information for today (April 27, 2026):

**Current Conditions:**
The current temperature is 57°F with sunny conditions and a RealFeel of 56°. There are winds from the ENE at 18 mph with gusts up to 23 mph.

**Today's Weather:**
Today (Monday, April 27) will feature increasingly windy conditions with brilliant sunshine. The high temperature will reach 65°F, while tonight will be breezy this evening then clear

🔧 web_search({'query': 'Istanbul weather today'})


<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `length`
- usage: `Usage(prompt_tokens=11418, completion_tokens=255, total_tokens=11673, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 11418, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 255, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

</details>

</div>

In [ ]:
test_eq(comp.tool_calls[0].server, True)

MCP (mocked):

In [ ]:
comp = mk_completion({"model": "claude-sonnet-4-20250514", "content": [{"type": "mcp_tool_use", "id": "mcptoolu_abc", "name": "get_weather", "input": {"city": "Istanbul"}, "server_name": "weather-server"}], "stop_reason": "tool_use", "usage": {"input_tokens": 50, "output_tokens": 30}}, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">



🔧 get_weather({'city': 'Istanbul'})


<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=50, completion_tokens=30, total_tokens=80, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 50, 'output_tokens': 30})`

</details>

</div>

In [ ]:
test_eq(comp.tool_calls[0].server, True)

### Streaming

Streaming module implements `PartAccum` and `acollect_stream` which are used to collate canonical `Delta` objects into a final `Completion` object similar to the non-streaming path while yielding text/thinking/citations. Each provider needs to implement `norm_sse_event` which will transform an event into a compatible `Delta` object which will be consumed by the streaming module

In [ ]:
class PrintStream:
    max_print = 10
    cnt = 0
    def print_stream(self,o,postproc=noop):
        if not isinstance(o, Completion): 
            if o.get('thinking') and self.cnt<self.max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            if citations:= o.get('citations'): print(postproc(citations),end='',flush=True)
            self.cnt+=1
        else: self.cnt = 0

In [ ]:
print_stream = PrintStream().print_stream

OpenAI responses api returns the whole tool call response with a `response.output_item.done` event which doesn't require argument collation and makes our job easy. If api wasn't returning the collated tool call, we'd have needed to provide delta arguments in Delta object's `tool_call` like:

```py
ToolCall(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, extra=extra)
```

In [ ]:
#| export
def norm_sse_event(ev, **kwargs):
    typ = ev.get("type")
    text, thinking, tcs, citations = None, None, [], None
    if typ == "content_block_start":
        cb = ev.get("content_block", {})
        if cb.get("type", "").endswith("_tool_result"): return Delta(server_tool_result=cb, raw=ev, **kwargs)
        if tc := norm_tool_call(cb):
            if not tc.arguments: tc.arguments = {'_delta': ''}
            tcs = [tc]
    elif typ == "content_block_delta":
        d = ev.get("delta", {})
        dtyp = d.get("type")
        if   dtyp == "text_delta": text = d.get("text")
        elif dtyp == "thinking_delta": thinking = d.get("thinking")
        elif dtyp == "input_json_delta":
            tcs = [ToolCall(id=str(ev.get("index", "")), name="", arguments={"_delta": d.get("partial_json", '')})]
        elif dtyp == "citations_delta": 
            citations = listify(d.get('citation',[]))
    elif typ == "message_delta":
        fin = norm_finish(ev.get('delta'))
        return Delta(finish_reason=fin, usage=norm_usage(ev), raw=ev, **kwargs)
    elif typ == "error": raise api_error_from_event(ev)
    return Delta(text=text, thinking=thinking, tool_calls=tcs, citations=citations, raw=ev, **kwargs)

Each provider also needs to implement `delta_index_fn` which returns a current idx and last idx and used by `PartAccum` like `part_accum.append(typ, idx, **tc_kwargs)`.

Here `d` is a `Delta` object, `typ` is the name of the event, it also takes last event's name and it's index to decide how to do collation

Anthropic events from the same group (e.g. same tool call parts) share the same index

In [ ]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    return nested_idx(d, 'raw', 'index'), None

Now we can create and register openai responses api's `acollect_stream`:

In [ ]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='anthropic', **kwargs)
    async for o in res: yield o

In [ ]:
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, finalize_usage=finalize_usage, acollect_stream=acollect_stream)

The following tests demonstrate completions with streaming and non-streaming paths

#### Thinking

In [ ]:
msgs, eff = [{"role": "user", "content": "Say hi"}], {"type": "enabled", "budget_tokens": 5000}
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user is asking me to say hi, which is a simple greeting. I should respond in a friendly and warm manner.

</details>

Hi there! How are you doing today?

<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=38, completion_tokens=12, total_tokens=83, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=33, raw={'input_tokens': 38, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 45, 'service_tier': 'standard', 'inference_geo': 'not_available'})`

</details>

</div>

In [ ]:
test_eq(comp.usage.reasoning_tokens > 0, True)
test_eq(comp.usage.completion_tokens + comp.usage.reasoning_tokens, comp.usage.raw['output_tokens'])


In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=msgs,max_tokens=8000,thinking=eff,stream=True)
async for o in acollect_stream(resp): print_stream(o)

🧠

🧠

Hi there! How are you doing today?

In [ ]:
test_eq(o.usage.reasoning_tokens > 0, True)
test_eq(o.usage.completion_tokens + o.usage.reasoning_tokens, o.usage.raw['output_tokens'])


In [ ]:
o

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

This is a simple, friendly greeting. I should respond in a warm and welcoming way.

</details>

Hi there! How are you doing today?

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=38, completion_tokens=17, total_tokens=77, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=22, raw={'input_tokens': 38, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 39})`

</details>

</div>

In [ ]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [ ]:
msgs = [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user is asking me to calculate two addition problems:
1. 3 + 5
2. 10 + 20

They specifically want me to use the simple_add tool and do this in parallel. I have access to the simple_add function which takes parameters 'a' and 'b' (both integers). I can make both function calls simultaneously.

For the first calculation: a=3, b=5
For the second calculation: a=10, b=20

I have all the required parameters for both calls, so I can proceed.

</details>

I'll calculate both additions for you using the simple_add tool in parallel.

🔧 simple_add({'a': 3, 'b': 5})



🔧 simple_add({'a': 10, 'b': 20})


<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=447, completion_tokens=152, total_tokens=717, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=118, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 270, 'service_tier': 'standard', 'inference_geo': 'not_available'})`

</details>

</div>

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff, stream=True)
async for o in acollect_stream(resp): print_stream(o)

🧠

🧠

🧠

🧠

🧠

I'll calculate both additions for you using the simple_add tool in parallel:

In [ ]:
o.raw['deltas']

[Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None, raw={'type': 'message_start', 'message': {'model': 'claude-sonnet-4-20250514', 'id': 'msg_01V8jkRhFs2puUgrS4F3rcPC', 'type': 'message', 'role': 'assistant', 'content': [], 'stop_reason': None, 'stop_sequence': None, 'stop_details': None, 'usage': {'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 7, 'service_tier': 'standard', 'inference_geo': 'not_available'}}}),
 Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None, raw={'type': 'content_block_start', 'index': 0, 'content_block': {'type': 'thinking', 'thinking': '', 'signature': ''}}),
 Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_r

In [ ]:
def test(o:dict, nm:str):
    'echo dict'
    return o

Use &`test` and pass a dict with key value pairs

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Sure! Also, heads up: the `bash` tool isn't available — looks like it hasn't been run yet.

Let me call `test` with a sample dict:

<details class='tool-usage-details' markdown='1'>
<summary><code>test(o=&quot;{&#x27;name&#x27;: &#x27;hello&#x27;, &#x27;count&#x27;: 42, &#x27;active&#x27;:…&quot;)→&quot;{&#x27;name&#x27;: &#x27;hello&#x27;, &#x27;count&#x27;: 42, &#x27;active&#x27;:…&quot;</code></summary>

```json
{
  "id": "toolu_01HGzybRb6JjAFnrjGMf7wGQ",
  "server": false,
  "call": {
    "function": "test",
    "arguments": {
      "o": "{'name': 'hello', 'count': 42, 'active': True}"
    }
  },
  "result": "{'name': 'hello', 'count': 42, 'active': True}"
}
```

</details>

It echoed back the dict as expected: `{'name': 'hello', 'count': 42, 'active': True}`.

<details class='token-usage-details' markdown='1'><summary>$0.1396</summary>

`total=88,097 | in=87,922 | out=151 | cached=33.2% | cache_new=43,880 | reasoning=24 | $0.1396`

</details>


In [ ]:
sch = lite_mk_func(test)
test_tools = [{"name": "test", "description": "echo dict", "input_schema": sch['function']['parameters']}]
msgs = [{"role": "user", "content": "Use test and pass a dict with key value pairs as the first param, then also pass a random string to `nm`"}]

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=test_tools, max_tokens=8000, thinking=eff, stream=True)
async for o in acollect_stream(resp): print_stream(o)

🧠

🧠

🧠

🧠

🧠

🧠

I'll call the test function with a dictionary containing key-value pairs and a random string for

 the name parameter.

In [ ]:
o.raw['deltas']

[Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None, raw={'type': 'message_start', 'message': {'model': 'claude-sonnet-4-20250514', 'id': 'msg_01BEpSDPoauR7rs6EPF1uQw1', 'type': 'message', 'role': 'assistant', 'content': [], 'stop_reason': None, 'stop_sequence': None, 'stop_details': None, 'usage': {'input_tokens': 444, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 6, 'service_tier': 'standard', 'inference_geo': 'not_available'}}}),
 Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_reason=None, usage=None, raw={'type': 'content_block_start', 'index': 0, 'content_block': {'type': 'thinking', 'thinking': '', 'signature': ''}}),
 Delta(text=None, thinking=None, refusal='', tool_calls=[], citations=None, server_tool_result=None, finish_r

In [ ]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name); test_eq(tc1.arguments,tc2.arguments); test_eq(tc1.server,tc2.server)

In [ ]:
test_tcs(comp.tool_calls,o.tool_calls)

In [ ]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=447, completion_tokens=152, total_tokens=717, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=118, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 270, 'service_tier': 'standard', 'inference_geo': 'not_available'}),
 Usage(prompt_tokens=447, completion_tokens=153, total_tokens=705, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=105, raw={'input_tokens': 447, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 258}))

#### Web search

In [ ]:
def add_ant_link(citations):
    links = []
    for c in citations:
        if 'title' not in c: return ''
        title = c['title'].replace('"', '\\"')
        links.append(f'[*]({c["url"]} "{title}")')
    return ' '.join(links)

In [ ]:
msgs = [{"role": "user", "content": "Use web search to get the weather in Istanbul?"}]
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user is asking for current weather information in Istanbul. This requires real-time data that changes frequently (weather conditions), so this is a good case for using web search. I should search for current weather in Istanbul.

</details>

Based on the current weather information for Istanbul:

**Current Weather:**
Istanbul is currently mostly clear with a temperature of 46°F. The humidity is at 74% with winds from the SSW at 5 mph. The RealFeel temperature is 45°F, which feels chilly.

**Today's Forecast:**
Today's low is expected to be 46°F with clear conditions. Tomorrow's forecast shows mostly sunny skies with a high of 65°F. There is only a 1% chance of precipitation.

**Additional Details:**
- Cloud cover is at 21% with visibility of 15 miles
- Air quality is currently unhealthy
- A jacket or sweater is recommended due to the chilly conditions

Overall, Istanbul is experiencing clear and cool weather with no significant precipitation expected.

🔧 web_search({'query': 'Istanbul weather today'})


<details markdown='1'>

- model: `claude-sonnet-4-20250514`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11098, completion_tokens=383, total_tokens=11536, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=55, raw={'input_tokens': 11098, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 438, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

</details>

</div>

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=eff, stream=True)
async for o in acollect_stream(resp): print_stream(o, add_ant_link)

🧠

🧠

🧠

Based

 on the current weather information for Istanbul, here's what I found:

**Current Weather in

...

Rain is expected Thursday evening through Friday

 morning

, but today remains mostly clear and pleasant.

The

 weather is quite nice in Istanbul today, with comfortable temperatures and sunny skies -

 perfect for outdoor activities!

In [ ]:
o

<div class="prose" markdown="1">

<details><summary>Thinking</summary>

The user is asking for the weather in Istanbul. This requires real-time/current information, so I should use web search. I'll search for current weather conditions in Istanbul.

</details>

Based on the current weather information for Istanbul, here's what I found:

**Current Weather in Istanbul:**

Today (April 29th) in Istanbul is mostly sunny with a high of 66°F (19°C), and tonight will be partly cloudy with a low of 47°F (8°C). The current temperature is 64°F (18°C), with a RealFeel temperature of 69°F and sunny conditions.

**Current Conditions:**
- Wind: East-northeast at 8 mph with gusts up to 9 mph
- Air Quality: Fair - generally acceptable for most individuals

**Forecast:**
Rain is expected Thursday evening through Friday morning, but today remains mostly clear and pleasant.

The weather is quite nice in Istanbul today, with comfortable temperatures and sunny skies - perfect for outdoor activities!

🔧 web_search({'query': 'Istanbul weather today'})


<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=11041, completion_tokens=334, total_tokens=11415, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=40, raw={'input_tokens': 11041, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 374, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}})`

</details>

</div>

In [ ]:
test_tcs(comp.tool_calls, o.tool_calls)
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=11098, completion_tokens=383, total_tokens=11536, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=55, raw={'input_tokens': 11098, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 438, 'service_tier': 'standard', 'inference_geo': 'not_available', 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}),
 Usage(prompt_tokens=11041, completion_tokens=334, total_tokens=11415, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=40, raw={'input_tokens': 11041, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 374, 'server_tool_use': {'web_search_requests': 1, 'web_fetch_requests': 0}}))

## Denormalize

Helpers to transform internal representation types back to API payload

### Msgs

In [ ]:
def mk_user_msg(txt): return Msg(role='user', content=[Part(type=PartType.text, text=txt)])

Helpers to translate back to provider specific inputs from our internal `Msg` representation

In [ ]:
#| export
def _ant_cc(block, p):
    "Add cache_control to block if present in Part.data."
    if (cc := (p.data or {}).get('cache_control')): block['cache_control'] = cc
    return block

This is not exported -- media tool results are added later

In [ ]:
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    d = p.data or {}
    return _ant_cc(dict(type='tool_result', tool_use_id=d.get('id'), content=str(p.text)), p)

This is not exported -- media inputs are added later

In [ ]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = [_ant_cc(dict(type='text', text=p.text or ''), p) for p in m.content if p.type == PartType.text]
    return dict(role='user', content=parts)

In [ ]:
#| export
def denorm_tool_use(p:Part):
    "Convert canonical tool_use Part to Anthropic tool_use content block."
    d = p.data or {}
    block = dict(type='tool_use', id=d.get('id',''), name=d.get('name',''), input=d.get('arguments') or {})
    if 'caller' in d: block['caller'] = d['caller']
    return _ant_cc(block, p)

def denorm_assistant(m:Msg):
    "Convert canonical assistant Msg to Anthropic assistant message + synthetic tool results for non-Anthropic server tools."
    blocks, srv_results, srv_call_id = [], [], None
    for p in m.content:
        if p.type == PartType.thinking:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            elif sig:=(p.data or {}).get('signature',''): blocks.append(dict(type='thinking', thinking=p.text or '', signature=sig))
            else:                               blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.text:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            else: blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.tool_use:
            if p.data.get('server') and (p.data.get('id','') or '').startswith('srvtoolu_'):
                blocks.append(dict(type='server_tool_use', id=p.data['id'], name=p.data.get('name',''), input=p.data.get('arguments') or {}))
            elif p.data.get('server'):
                blocks.append(denorm_tool_use(p))
                srv_call_id = p.data.get('id','')
            else: blocks.append(denorm_tool_use(p))
        elif p.type == PartType.server_tool_result: blocks.append(p.data if p.data else dict(type='server_tool_result'))
    res = [dict(role='assistant', content=blocks)]
    if srv_results: res.append(dict(role='user', content=srv_results))
    return res

def denorm_tool(m:Msg):
    "Convert canonical tool Msg to Anthropic user message with tool_result blocks."
    blocks = [denorm_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return [dict(role='user', content=blocks)]

def denorm_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Anthropic messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_user(m))
        elif m.role == 'assistant': res.extend(denorm_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_tool(m))
    return res

### Tool Schemas

In [ ]:
#| export
def denorm_tool_schs(tools):
    "Convert canonical tools to Anthropic format."
    out = []
    for t in tools:
        fn = fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(name=name, description=desc, input_schema=params))
    return out

In [ ]:
test_eq(denorm_tool_schs([sch]), [{'name': 'simple_add', 'description': 'add numbers', 'input_schema': sch['function']['parameters']}])
test_eq(denorm_tool_schs([{"type": "web_search_20250305", "name": "web_search"}]), [{"type": "web_search_20250305", "name": "web_search"}])

### Tool Choice

In [ ]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to Anthropic format."
    if v is None: return None
    if v in ('auto',):                    return {'type': 'auto'}
    if v in ('required', 'any', 'force'): return {'type': 'any'}
    if v in ('none', 'off', 'disabled'):  return {'type': 'none'}
    return {'type': 'tool', 'name': v}

Modes

In [ ]:
test_eq(denorm_tool_choice('auto'), {'type': 'auto'})
test_eq(denorm_tool_choice('required'), {'type': 'any'})
test_eq(denorm_tool_choice('none'), {'type': 'none'})

Tool name

In [ ]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'tool', 'name': 'simple_add'})

None

In [ ]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

No `disable` option maybe add later if needed. Anthropic uses the newer adaptive thinking which will error with legacy models < 4.6

In [ ]:
#| export
def denorm_reasoning(v):
    "Map canonical reasoning_effort to Anthropic adaptive thinking + output_config."
    mp = dict(minimal='low', low='low', medium='medium', high='high', xhigh='xhigh', max='max')
    err = ValueError(f"Invalid reasoning effort for Anthropic: {v}, accepted string values are: {list(mp)} and dicts are passthrough")
    if v is None: return None
    elif isinstance(v, dict): return v
    elif isinstance(v, str) and v in mp:  return {"thinking": {"type": "adaptive"}, "output_config": {"effort": mp.get(v)}}
    raise err

In [ ]:
test_eq(denorm_reasoning('low'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "low"}})
test_eq(denorm_reasoning('high'), {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}})
test_eq(denorm_reasoning(None), None)

In [ ]:
denorm_reasoning(dict(thinking={"type":"adaptive", "display":"summarized"}, output_config={"effort":'high'}))

{'thinking': {'type': 'adaptive', 'display': 'summarized'},
 'output_config': {'effort': 'high'}}

### Web Search Options

In [ ]:
#| export
def denorm_web_search(v):
    "Map canonical web_search_options to Anthropic hosted web_search tool."
    _max_uses = {"low": 1, "medium": 5, "high": 10}
    t = {"type": "web_search_20250305", "name": "web_search"}
    if (typ := (v or {}).get("type")): t["type"] = typ
    if (s := (v or {}).get("search_context_size")):
        t["max_uses"] = _max_uses.get(s, 5)
    if (u := (v or {}).get("user_location", {}).get("approximate")):
        t["user_location"] = {"type": "approximate", **u}
    return t

None / empty

In [ ]:
test_eq(denorm_web_search(None), {"type": "web_search_20250305", "name": "web_search"})

search_context_size

In [ ]:
test_eq(denorm_web_search({"search_context_size": "medium"})["max_uses"], 5)

user_location

In [ ]:
loc = {"approximate": {"city": "Istanbul"}}
test_eq(denorm_web_search({"user_location": loc})["user_location"], {"type": "approximate", "city": "Istanbul"})

### System

In [ ]:
#| export
def denorm_system(sp):
    if isinstance(sp, Msg): sp = sp.content[0]
    if isinstance(sp, Part):
        block = dict(type='text', text=sp.text)
        if (cc := (sp.data or {}).get('cache_control')): block['cache_control'] = cc
        return [block]
    else: return sp

In [ ]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

Anthropic supports cache control and passes sp to `system`

In [ ]:
cc = {"cache_control": {"type": "ephemeral"}}
ant_sp = Part(PartType.text, sp*200, data=cc)

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,system=denorm_system(ant_sp), messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

<div class="prose" markdown="1">

Ahoy matey, I be a landlubber's digital parrot, ready to help ye with yer questions!

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=3811, completion_tokens=29, total_tokens=3840, cached_tokens=0, cache_creation_tokens=3801, reasoning_tokens=0, raw={'input_tokens': 10, 'cache_creation_input_tokens': 3801, 'cache_read_input_tokens': 0, 'output_tokens': 29})`

</details>

</div>

In [ ]:
test_eq(comp.usage.cache_creation_tokens>0, True)

### Media Inputs

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

**Canonical Part Types**

| `Part.type` | `Part.text` | Description |
|---|---|---|
| `input_image` | URL or `data:image/...;base64,...` | Image input — JPEG, PNG, GIF, WEBP |
| `input_audio` | `data:audio/...;base64,...` | Audio input — WAV, MP3 (base64 only for OpenAI) |
| `input_video` | URL | Video input — MP4, WebM (Gemini only) |
| `input_file` | URL or `data:application/...;base64,...` | Document input — PDF, DOCX, TXT, etc. |

**Provider Wire Formats**

**OpenAI Responses** ([InputImageContent](https://developers.openai.com/api/reference/resources/responses/methods/create), [InputFileContent](https://developers.openai.com/api/reference/resources/responses/methods/create) — `specs/openai.with-code-samples.yml:68361,68429`):
- Image: `{"type": "input_image", "image_url": url_or_data_url}`
- File: `{"type": "input_file", "file_url": url}` or `{"type": "input_file", "file_data": data_url, "filename": "..."}`
- Audio/Video: not supported

**OpenAI Chat** ([ContentPartImage](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartAudio](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartFile](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) — `specs/openai.with-code-samples.yml:35185,35217,35253`):
- Image: `{"type": "image_url", "image_url": {"url": url_or_data_url}}`
- Audio: `{"type": "input_audio", "input_audio": {"data": b64, "format": "wav"|"mp3"}}` — base64 only, WAV/MP3 only
- File: `{"type": "file", "file": {"file_data": data_url, "filename": "..."}}` — base64/file_id only, no URL
- Video: not supported

**Anthropic** ([RequestImageBlock](https://docs.anthropic.com/en/api/messages), [RequestDocumentBlock](https://docs.anthropic.com/en/api/messages) — `specs/anthropic.yml:12159,6740`):
- Image: `{"type": "image", "source": {"type": "url", "url": ...}}` or `{"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}}`
- File: `{"type": "document", "source": ...}` — same source pattern, PDF only (`media_type: "application/pdf"`)
- Audio/Video: not supported
- Supports `cache_control` on all content blocks

**Gemini** ([Part/Blob/FileData](https://ai.google.dev/api/generate-content) — `specs/gemini.json:189–387`):
- All media: `{"inlineData": {"mimeType": mime, "data": b64}}` or `{"fileData": {"mimeType": mime, "fileUri": url}}`
- Broadest format support — images, audio, video (incl. YouTube URLs), documents
- Supports `videoMetadata` for video start/end offsets

**Provider Support Matrix**

| Canonical type | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| `input_image` (URL) | ✅ | ✅ | ✅ | ✅ |
| `input_image` (base64) | ✅ | ✅ | ✅ | ✅ |
| `input_audio` (base64) | ❌ | ✅ (WAV/MP3) | ❌ | ✅ |
| `input_video` (URL) | ❌ | ❌ | ❌ | ✅ |
| `input_file` (URL) | ✅ | ❌ | ✅ | ✅ |
| `input_file` (base64) | ✅ | ✅ | ✅ | ✅ |


In [ ]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text: parts.append(_ant_cc({"type":"text", "text":p.text or ''}, p))
        elif p.type == PartType.input_image: parts.append(_ant_cc(denorm_image(p), p))
        elif p.type == PartType.input_audio: raise ValueError("Anthropic does not support audio input")
        elif p.type == PartType.input_video: raise ValueError("Anthropic does not support video input")
        elif p.type == PartType.input_file:  parts.append(_ant_cc(denorm_file(p), p))
    return dict(role='user', content=parts)

#### Image

In [ ]:
#| export
def denorm_image(p):
    if (b64:=data_url(p.text)): return {"type": "image", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "image", "source": {"type": "url", "url": p.text}}

Image URL

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

<div class="prose" markdown="1">

This image shows a serene mountain lake scene with a wooden dock or platform in the foreground. The lake has perfectly still water that creates beautiful mirror-like reflections of the surrounding landscape. In the background, there are snow-capped mountains, dense forests of evergreen trees, and a partly cloudy sky. The wooden planks in the foreground appear to be part of a dock, deck, or viewing platform from which this peaceful lakeside vista can be enjoyed. The overall composition creates a tranquil, scenic atmosphere typical of alpine or mountain wilderness areas.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=610, completion_tokens=119, total_tokens=729, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 610, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 119})`

</details>

</div>

Image data

In [ ]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

<div class="prose" markdown="1">

I can see that you've pointed to a specific pixel with a red arrow, but I'm not able to determine what color that individual pixel is from this image. The resolution and compression of the image make it difficult to identify the exact color of a single pixel. 

If you could provide more context about what you're trying to identify or perhaps zoom in on the area you're interested in, I'd be happy to help describe the colors I can observe in that region.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=18, completion_tokens=99, total_tokens=117, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 18, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 99})`

</details>

</div>

#### Files

OpenAI requires a filename, Anthropic requires the media type and Gemini requires the mime type

In [ ]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return {"type": "document", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "document", "source": {"type": "url", "url": p.text}}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_msgs([msg1]),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

<div class="prose" markdown="1">

The title of this paper is "Attention Is All You Need".

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=34586, completion_tokens=17, total_tokens=34603, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 34586, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 17})`

</details>

</div>

### Media Tool Call Results

A tool call can return an `input_image` or `input_file`

In [ ]:
#| export
def denorm_tool_result(p:Part):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    d = p.data or {}
    tid = d.get('id') or d.get('call_id','')
    if isinstance(p.text, list):
        blocks = []
        for pp in p.text:
            if   pp.type == PartType.text:        blocks.append({"type": "text", "text": pp.text or ""})
            elif pp.type == PartType.input_image: blocks.append(denorm_image(pp))
            elif pp.type == PartType.input_file:  blocks.append(denorm_file(pp))
            else: raise ValueError(f"Anthropic tool_result does not support {pp.type}")
        return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=blocks), p)
    return _ant_cc(dict(type='tool_result', tool_use_id=tid, content=str(p.text)), p)

In [ ]:
msgs = [Msg(role='user', content=[Part(type=PartType.text, text="What's in this image?")]),
        Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': '_test', 'name': 'test_img', 'arguments': {}, 'server': False})]),
        Msg(role='tool', content=[Part(type=PartType.tool_result, text=[Part(type=PartType.input_image, text=img_b64)], data={'id': '_test', 'name': 'test_img'})])]

In [ ]:
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_msgs(msgs),max_tokens=8000,stream=True)
async for comp in acollect_stream(resp): pass
comp

<div class="prose" markdown="1">

I can see a small red square or rectangle in the image. It appears to be a simple geometric shape with a solid red color against what looks like a white or transparent background. The image is quite minimal and doesn't contain any other visible elements, text, or details.

<details markdown='1'>

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=77, completion_tokens=58, total_tokens=135, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, raw={'input_tokens': 77, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 58})`

</details>

</div>

## Payload

Function to create the payload:

In [ ]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model, messages=denorm_msgs(msgs), max_tokens=kwargs.get('max_tokens') or 1024)
    if kwargs.get('stream'):                payload['stream'] = True
    if sp:=kwargs.get('system'):            payload['system'] = denorm_system(sp)
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if thk:=kwargs.get('reasoning_effort'): payload.update(denorm_reasoning(thk))
    if (wopts:=kwargs.get('web_search_options')) is not None: 
        payload.setdefault('tools', []).append(denorm_web_search(wopts))
    if (temp:=kwargs.get('temperature')) is not None: 
        payload['temperature'] = temp
    return payload

Function to create the default headers:

In [ ]:
#| export
def get_hdrs(api_key=None):
    return {"x-api-key": get_api_key(api_key, 'ANTHROPIC_API_KEY'), "anthropic-version": "2023-06-01"}

## Cost

A function to calculate api Completion cost from raw usage data, and model metadata returned by `get_model_info`

In [ ]:
#| export
def cost(usage, m):
    raw = usage.raw
    in_tok = raw['input_tokens']
    cache_read = raw.get('cache_read_input_tokens', 0)
    cc = raw.get('cache_creation', {}) or {}
    cache_5m  = cc.get('ephemeral_5m_input_tokens', 0)
    cache_1h  = cc.get('ephemeral_1h_input_tokens', 0)
    cost  = in_tok     * m.input_cost_per_token
    cost += raw['output_tokens'] * m.output_cost_per_token
    cost += cache_read * m.get('cache_read_input_token_cost', 0)
    cost += cache_5m   * m.get('cache_creation_input_token_cost', 0)
    cost += cache_1h   * m.get('cache_creation_input_token_cost_above_1hr', 0)
    return cost

## Register API

Register the final API namespace:

- `norm_tool_calls`, `norm_parts`, `norm_finish`, `norm_usage` required during the Completion object construction.
- `mk_payload` is used to construct the api request payload
- `accollect_stream` is used in streaming api call path
- `get_hdrs` is used to create the deafult request headers
- `op_path` define the non-streaming and streaming OpenAPIClient op names

In [ ]:
#| export
api_registry.register('anthropic', norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, acollect_stream=acollect_stream, mk_payload=mk_payload, cost=cost, get_hdrs=get_hdrs,
    op_path=('messages.messages_post','messages.messages_post'))

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()